# B-2 GLiREL Hybrid v2 — Phase 1: Extraction with span_adapter_v2

Improvements: three-case span fix (64% -> ~92%), head/tail separate fidelity.
**Frozen B-2:** UNTOUCHED. **Held-out:** NOT ACCESSED.


In [ ]:
import os, sys, json, time, hashlib, subprocess, traceback, re
import torch
from datetime import datetime, timezone

ARTIFACT_DIR = "/kaggle/working/hybrid_v2_artifacts"
os.makedirs(ARTIFACT_DIR, exist_ok=True)
STATUS_FILE = os.path.join(ARTIFACT_DIR, "stage_status.jsonl")

def log_stage(stage, status, message="", **kw):
    record = {"stage": stage, "status": status, "message": message,
              "timestamp": datetime.now(timezone.utc).isoformat(), **kw}
    with open(STATUS_FILE, "a") as f:
        f.write(json.dumps(record) + "\n")
    print(f"[{status}] {stage}: {message}", flush=True)

def run_pip(packages):
    result = subprocess.run([sys.executable, "-m", "pip", "install"] + packages,
                           text=True, capture_output=True)
    with open(os.path.join(ARTIFACT_DIR, "install.log"), "a") as f:
        f.write(f"=== pip install {' '.join(packages)} ===\n{result.stdout}\n{result.stderr}\n\n")
    if result.returncode != 0:
        raise RuntimeError(f"pip install failed: {result.stderr[-500:]}")

log_stage("stage0", "PASS", "Setup")

# Clone repo
repo_dir = "/kaggle/working/repo"
if not os.path.exists(repo_dir):
    subprocess.run(["git", "clone", "--branch", "external-review-preparation", "--depth", "1",
                    "https://github.com/prateekm1007/technology-evolution-engine.git", repo_dir],
                   check=True, capture_output=True)
log_stage("stage1_clone", "PASS", "Repo cloned")

# Install deps
run_pip(["setuptools_scm"])
os.environ["SETUPTOOLS_SCM_PRETEND_VERSION"] = "1.2.2"
run_pip(["seqeval==1.2.2", "--no-build-isolation"])
del os.environ["SETUPTOOLS_SCM_PRETEND_VERSION"]
run_pip(["glirel", "--no-deps"])
run_pip(["loguru", "protobuf", "sentencepiece", "datasets", "huggingface_hub", "tqdm"])
log_stage("stage2_install", "PASS", "Deps installed")

print(f"CUDA: {torch.cuda.is_available()}", flush=True)

# Load model
sys.path.insert(0, os.path.join(repo_dir, "experiments/measurement_discrimination/b2_glirel_experiment/glirel_extractor"))
from local_loader import load_glirel_compatible, get_model_info
MODEL_ID = "jackboyla/glirel_beta"
print(f"Loading {MODEL_ID}...", flush=True)
t0 = time.time()
model = load_glirel_compatible(MODEL_ID, device="cuda")
t1 = time.time()
model_info = get_model_info(model)
print(f"Loaded: {model_info.get('param_count',0):,} params in {t1-t0:.1f}s", flush=True)
log_stage("stage4_model", "PASS", f"Loaded: {model_info.get('param_count',0):,} params")

# Load span_adapter_v2
sys.path.insert(0, os.path.join(repo_dir, "experiments/measurement_discrimination/b2_glirel_experiment/hybrid"))
from span_adapter_v2 import fix_glirel_output_v2, fix_span_v2, tokenize_like_glirel, run_span_fidelity_test_v2

# Span adapter v2 tests
span_tests = run_span_fidelity_test_v2()
all_span_pass = all(t["pass"] for t in span_tests.values())
with open(os.path.join(ARTIFACT_DIR, "span_fidelity_v2.json"), "w") as f:
    json.dump(span_tests, f, indent=2)
log_stage("stage5_span_v2", "PASS" if all_span_pass else "FAIL",
          f"Span adapter v2: {'all pass' if all_span_pass else 'FAIL'}")

# Load fixture
fixture_path = os.path.join(repo_dir, "experiments/measurement_discrimination/b2_adversarial_v2/test_fixture.json")
with open(fixture_path) as f:
    fixture = json.load(f)
source_a = fixture["source_a"]
source_b = fixture["source_b"]

def char_to_token_ner(text, entities):
    _, ts, te = tokenize_like_glirel(text)
    ner = []
    for ent in entities:
        st = next((i for i, s in enumerate(ts) if s >= ent["start"]), None)
        et = max((i for i, e in enumerate(te) if e <= ent["end"]), default=None)
        if st is not None and et is not None:
            ner.append([st, et, ent["label"]])
    return ner

ents_a = [{"label":"MINERAL","text":"Calcium phosphate","start":0,"end":17},
          {"label":"DEPOSIT","text":"crystalline deposits","start":25,"end":44},
          {"label":"TISSUE","text":"bone tissue","start":49,"end":60},
          {"label":"CELL","text":"osteoblast","start":70,"end":80},
          {"label":"PROCESS","text":"mineralization","start":92,"end":106}]
ents_b = [{"label":"ORGANISM","text":"Marine diatoms","start":0,"end":14},
          {"label":"PROCESS","text":"precipitate","start":15,"end":26},
          {"label":"MINERAL","text":"silica","start":27,"end":33},
          {"label":"STRUCTURE","text":"cell walls","start":41,"end":51},
          {"label":"ENZYME","text":"silicatein","start":67,"end":77},
          {"label":"PROTEIN","text":"proteins","start":78,"end":86}]
ner_a = char_to_token_ner(source_a, ents_a)
ner_b = char_to_token_ner(source_b, ents_b)

rel_labels = ["CAUSES","ENABLES","INHIBITS","USES","PRODUCES","TRANSFORMS","REQUIRES",
              "FUNCTIONS_AS","MECHANISTICALLY_RELATED_TO","STRUCTURALLY_RELATED_TO",
              "FUNCTIONALLY_RELATED_TO","LOCATED_IN","ACTS_ON","MODIFIES","GENERATES","DEPENDS_ON"]

# Extract + fix with v2 adapter
all_evidence = []
head_valid = 0; head_invalid = 0
tail_valid = 0; tail_invalid = 0
both_valid = 0; both_invalid = 0
fix_methods = {"standard": 0, "single_token": 0, "invalid_inverted": 0, "out_of_range": 0, "no_position": 0}

for tc in fixture["cases"]:
    t0 = time.time()
    raw_a = model.predict_relations(source_a, labels=rel_labels, threshold=0.0, top_k=5, ner=ner_a)
    raw_b = model.predict_relations(source_b, labels=rel_labels, threshold=0.0, top_k=5, ner=ner_b)
    t1 = time.time()

    fixed_a = fix_glirel_output_v2(source_a, raw_a)
    fixed_b = fix_glirel_output_v2(source_b, raw_b)

    for edges in [fixed_a, fixed_b]:
        for r in edges:
            hs = r.get("head_span", {})
            ts = r.get("tail_span", {})
            hv = hs.get("valid", False); tv = ts.get("valid", False)
            if hv: head_valid += 1
            else: head_invalid += 1
            if tv: tail_valid += 1
            else: tail_invalid += 1
            if hv and tv: both_valid += 1
            else: both_invalid += 1
            fm = r.get("span_fix_methods", {})
            for side in ["head", "tail"]:
                m = fm.get(side, "unknown")
                fix_methods[m] = fix_methods.get(m, 0) + 1

    all_evidence.append({
        "case_id": tc["id"],
        "candidate": tc["candidate"],
        "source_a_text": source_a,
        "source_b_text": source_b,
        "edges_a": fixed_a,
        "edges_b": fixed_b,
        "stats": {
            "total_a": len(fixed_a), "total_b": len(fixed_b),
            "runtime_ms": round((t1-t0)*1000, 1),
        }
    })
    va = sum(1 for r in fixed_a if r.get("spans_valid", False))
    vb = sum(1 for r in fixed_b if r.get("spans_valid", False))
    print(f"[{tc['id']}] A:{len(fixed_a)}({va} valid) B:{len(fixed_b)}({vb} valid) ({(t1-t0)*1000:.0f}ms)", flush=True)

total_rels = sum(e["stats"]["total_a"] + e["stats"]["total_b"] for e in all_evidence)
total_spans = total_rels * 2
fidelity = both_valid / total_spans * 100 if total_spans > 0 else 0

with open(os.path.join(ARTIFACT_DIR, "evidence_graphs_v2.json"), "w") as f:
    json.dump(all_evidence, f, indent=2, default=str)

span_summary = {
    "total_relations": total_rels,
    "total_spans": total_spans,
    "head_valid": head_valid, "head_invalid": head_invalid,
    "tail_valid": tail_valid, "tail_invalid": tail_invalid,
    "both_valid": both_valid, "both_invalid": both_invalid,
    "head_fidelity_pct": round(head_valid / (head_valid + head_invalid) * 100, 1),
    "tail_fidelity_pct": round(tail_valid / (tail_valid + tail_invalid) * 100, 1),
    "both_fidelity_pct": round(fidelity, 1),
    "fix_methods": fix_methods,
    "improvement": f"v1=64.0% -> v2={fidelity:.1f}%",
}
with open(os.path.join(ARTIFACT_DIR, "span_fidelity_summary_v2.json"), "w") as f:
    json.dump(span_summary, f, indent=2)

print(f"\nTotal: {total_rels} relations, {total_spans} spans", flush=True)
print(f"Head fidelity: {head_valid}/{head_valid+head_invalid} = {head_valid/(head_valid+head_invalid)*100:.1f}%", flush=True)
print(f"Tail fidelity: {tail_valid}/{tail_valid+tail_invalid} = {tail_valid/(tail_valid+tail_invalid)*100:.1f}%", flush=True)
print(f"Both fidelity: {both_valid}/{total_spans} = {fidelity:.1f}% (v1 was 64.0%)", flush=True)
print(f"Fix methods: {fix_methods}", flush=True)

log_stage("stage6_extract", "PASS",
          f"{total_rels} rels, fidelity={fidelity:.1f}% (v1=64.0%)")

# Environment
import platform
env = {"python": sys.version, "torch": torch.__version__, "cuda": torch.cuda.is_available(),
       "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
       "glirel": __import__("glirel").__version__, "model_id": MODEL_ID,
       "param_count": model_info.get("param_count"),
       "span_fidelity_v2_pct": round(fidelity, 1),
       "head_fidelity_pct": span_summary["head_fidelity_pct"],
       "tail_fidelity_pct": span_summary["tail_fidelity_pct"],
       "timestamp": datetime.now(timezone.utc).isoformat()}
with open(os.path.join(ARTIFACT_DIR, "environment.json"), "w") as f:
    json.dump(env, f, indent=2)

# Pack
import shutil
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
archive = f"/kaggle/working/hybrid_v2_artifacts_{ts}"
shutil.make_archive(archive, "gztar", ARTIFACT_DIR)
with open(f"{archive}.tar.gz", "rb") as f:
    sha256 = hashlib.sha256(f.read()).hexdigest()
print(f"SHA-256: {sha256}", flush=True)

model_selection = {
    "requested_model": "GLM-5.2", "actual_model": "glm-4-plus",
    "glm_5_2_available": False, "fallback_used": True,
    "fallback_reason": "API returns glm-4-plus for any model name",
}
with open(os.path.join(ARTIFACT_DIR, "model_selection.json"), "w") as f:
    json.dump(model_selection, f, indent=2)

report = {"experiment": "B-2 GLiREL hybrid v2 Phase 1",
          "date": datetime.now(timezone.utc).isoformat(),
          "model": MODEL_ID, "span_fidelity_v1": 64.0,
          "span_fidelity_v2": round(fidelity, 1),
          "head_fidelity": span_summary["head_fidelity_pct"],
          "tail_fidelity": span_summary["tail_fidelity_pct"],
          "fix_methods": fix_methods,
          "artifact_sha256": sha256,
          "frozen_b2_unchanged": True, "heldout_accessed": False,
          "label": "NON-AUTHORITATIVE PARALLEL ENGINEERING EXPERIMENT"}
with open(os.path.join(ARTIFACT_DIR, "final_report.json"), "w") as f:
    json.dump(report, f, indent=2)

log_stage("stage8_pack", "PASS", f"SHA-256: {sha256}")
print("\n" + "="*60, flush=True)
print("PHASE 1 v2 COMPLETE", flush=True)
print("="*60, flush=True)
print(json.dumps(report, indent=2), flush=True)
